# HuggingFaceLM Batched Prefetch Demo

`HuggingFaceLM.prefetch()` groups sibling states (children of the same parent) and evaluates them in a single batched forward pass instead of one call per child. This reduces model invocations and wall time when expanding multiple continuations from the same context.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import numpy as np
from transduction.util import set_memory_limit
from transduction.lm.huggingface_lm import HuggingFaceLM

set_memory_limit(8)

lm = HuggingFaceLM.from_name('gpt2')
print(f'Vocab size: {len(lm._decode)}')

## Setup: parent state and child tokens

We create a parent state by feeding in the token for `" The"`, then pick the top-10 next tokens as children to expand.

In [ ]:
# Encode " The" as a single token and advance to get a parent state
the_id = lm._encode[b' The']
parent = lm.initial() >> the_id

# Pick 10 likely continuations to expand
top10 = list(parent.logp_next.top(10).keys())
print('Children to expand:')
for tid in top10:
    print(f'  token {tid:5d}  {lm._decode[tid]!r:12s}  logp={parent.logp_next[tid]:.3f}')

## Sequential baseline

Create 10 children from `parent` and access `logp_next` on each. Each access triggers a separate model forward call.

In [ ]:
calls_before = lm._calls
t0 = time.perf_counter()

seq_children = [parent >> tid for tid in top10]
seq_logps = [child.logp_next[child.eos] for child in seq_children]  # force evaluation

seq_time = time.perf_counter() - t0
seq_calls = lm._calls - calls_before

print(f'Sequential: {seq_calls} forward calls in {seq_time:.3f}s')

## Batched prefetch

Same 10 children, but we call `lm.prefetch()` before accessing `logp_next`. The prefetch groups siblings and runs a single batched forward pass.

In [ ]:
calls_before = lm._calls
t0 = time.perf_counter()

batch_children = [parent >> tid for tid in top10]
lm.prefetch(batch_children)  # <-- single batched forward pass
batch_logps = [child.logp_next[child.eos] for child in batch_children]

batch_time = time.perf_counter() - t0
batch_calls = lm._calls - calls_before

print(f'Prefetch:   {batch_calls} forward calls in {batch_time:.3f}s')

## Correctness check

Sequential and prefetched logprobs must match exactly.

In [ ]:
for i, tid in enumerate(top10):
    assert np.isclose(seq_logps[i], batch_logps[i], atol=1e-5), \
        f'Mismatch at token {tid}: seq={seq_logps[i]:.6f} vs batch={batch_logps[i]:.6f}'

print('All logprobs match.')

## Timing comparison

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

labels = ['Sequential', 'Prefetch']

ax1.bar(labels, [seq_calls, batch_calls], color=['#d9534f', '#5cb85c'])
ax1.set_ylabel('Forward calls')
ax1.set_title('Model forward calls')
for i, v in enumerate([seq_calls, batch_calls]):
    ax1.text(i, v + 0.2, str(v), ha='center', fontweight='bold')

ax2.bar(labels, [seq_time, batch_time], color=['#d9534f', '#5cb85c'])
ax2.set_ylabel('Wall time (s)')
ax2.set_title('Wall time')
for i, v in enumerate([seq_time, batch_time]):
    ax2.text(i, v + 0.002, f'{v:.3f}s', ha='center', fontweight='bold')

fig.tight_layout()
plt.show()

print(f'Speedup: {seq_time/batch_time:.1f}x')

## Idiomatic pattern: prepare / prefetch / complete

In beam search (`CharacterBeam`, `GeneralizedBeam`), the typical three-phase loop is:

1. **Prepare** — create child states (`parent >> token`) for all expansions
2. **Prefetch** — call `lm.prefetch(children)` to batch siblings
3. **Complete** — access `logp_next` on each child (now cache-warm)

This decouples state construction from evaluation, letting the LM batch efficiently.

In [ ]:
# Simulate a beam search step with two active hypotheses expanding 5 tokens each.
hyp_a = lm.initial() >> lm._encode[b' The']
hyp_b = lm.initial() >> lm._encode[b' In']

tokens_a = list(hyp_a.logp_next.top(5).keys())
tokens_b = list(hyp_b.logp_next.top(5).keys())

# Phase 1: prepare
children = [hyp_a >> t for t in tokens_a] + [hyp_b >> t for t in tokens_b]

# Phase 2: prefetch (batches siblings of hyp_a together, siblings of hyp_b together)
calls_before = lm._calls
lm.prefetch(children)
prefetch_calls = lm._calls - calls_before

# Phase 3: complete — logp_next is already cached, no new forward calls
calls_before = lm._calls
scores = [c.logp_next[c.eos] for c in children]
complete_calls = lm._calls - calls_before

print(f'Prefetch phase:  {prefetch_calls} forward calls (2 batches, one per hypothesis)')
print(f'Complete phase:  {complete_calls} forward calls (all cached)')
print(f'Total children:  {len(children)}')